## Shared Config (if needed)

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from matplotlib.ticker import MultipleLocator

from ravelry_scraper.constants import DATA_DIR


In [ ]:
con = duckdb.connect()

parquet_files = list(DATA_DIR.glob("**/*.parquet"))

# Register Parquet datasets as SQL views
for file in parquet_files:
    name = f"{file.parent.name}_{file.stem}"
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{file}')
    """)


In [ ]:
# Store list of all DuckDB views
duckdb_views = [row[0] for row in con.execute("""
    SELECT view_name
    FROM duckdb_views
""").fetchall()]

# Print all DuckDB views to verify the list
print(duckdb_views)

In [ ]:
pl.Config.set_fmt_float("full")
pl.Config.set_tbl_rows(25)
pl.Config.set_thousands_separator(",")

# Designers (aka PatternAuthors)

Ravelry's API exposes an endpoint for `GET /designers/{id}.json` that returns Designer (aka PatternAuthor) records. A Designer may be associated with more than 1 User (but typically is only associated to 1). Cleaning in Python focused on dropping unused columns, standardizing data types, and extracting nested records (e.g., Users) into separate tables, then storing just the IDs in their place. From the cleaned data, notable columns include:

| **Notable Fields** | **Use Cases** |
| --- | --- |
| `id`, `name`, `user_ids` | Identify Designers and join them to Users |
| `patterns_count`, `crochet_pattern_count`, `knitting_pattern_count` | Evaluate Designer productivity and craft preference |
| `favorites_count` | Analyze Designer popularity |

### Documentation
- https://www.ravelry.com/api#designers_show
- https://www.ravelry.com/api#PatternAuthor_full_result

## Data Cleaning

First, we can confirm that cleaning the data in Python did not remove any records:

In [ ]:
display(
    con.sql("""
        SELECT
            (SELECT COUNT(*) FROM raw_designers) AS raw_count,
            (SELECT COUNT(*) FROM clean_designers) AS clean_count,
            (SELECT (SELECT COUNT(*) FROM raw_designers) - (SELECT COUNT(*) FROM clean_designers)) AS difference
    """).pl()
)

Second, we can compare the two schemas to see which columns were dropped and how data types were standardized and / or extracted into separate tables:

In [ ]:
display(
    con.sql("""
        WITH raw_schema AS (
            DESCRIBE raw_designers
        ),
        clean_schema AS (
            DESCRIBE clean_designers
        )
        SELECT 
            COALESCE(raw_schema.column_name, clean_schema.column_name) AS column_name,
            CASE
                WHEN raw_schema.column_type LIKE 'STRUCT%' THEN 'STRUCT()'
                ELSE COALESCE(raw_schema.column_type, '<EXTRACTED>')
            END AS raw_type,
            CASE
                WHEN raw_schema.column_type LIKE 'STRUCT%' THEN '<EXTRACTED>'
                ELSE COALESCE(clean_schema.column_type, '<DROPPED>')
            END AS clean_type,
        FROM raw_schema
        FULL OUTER JOIN clean_schema
        ON raw_schema.column_name = clean_schema.column_name
    """).df()
)


By viewing a few lines, we can get a better sense for the dataset we weill be working with:

In [ ]:
display(
    con.sql("""
        SELECT * 
        FROM clean_designers
        LIMIT 10
    """).pl()
)

## Data Exploration

TODO - describe

### Summary Statistics

In [ ]:
designer_summary_df = con.sql("""
    WITH averages AS (
        SELECT
            COUNT(*) AS num_designers,
            ROUND(
                AVG(patterns_count)
            , 2) AS avg_num_patterns,
            MEDIAN(patterns_count) AS median_num_patterns,
            MAX(patterns_count) AS max_num_patterns,
            ROUND(
                AVG(knitting_pattern_count)
            , 2) AS avg_num_knitting_patterns,
            MEDIAN(knitting_pattern_count) AS median_num_knitting_patterns,
            MAX(knitting_pattern_count) AS max_num_knitting_patterns,
            ROUND(
                AVG(crochet_pattern_count)
            , 2) AS avg_num_crochet_patterns,
            MEDIAN(crochet_pattern_count) AS median_num_crochet_patterns,
            MAX(crochet_pattern_count) AS max_num_crochet_patterns,
            MAX(LENGTH(user_ids)) AS max_user_ids_length
        FROM clean_designers
    )
    SELECT * FROM averages
""").pl()

display(designer_summary_df.transpose(include_header=True))

TODO: Describe summary stats

### Most Popular Designers

In [ ]:
con.sql("""
    SELECT
        name,
        favorites_count
    FROM clean_designers
    ORDER BY favorites_count DESC
    LIMIT 10
""").pl()

### Most Prolific Designers

In [ ]:
con.sql("""
    SELECT
        name,
        patterns_count
    FROM clean_designers
    ORDER BY patterns_count DESC
    LIMIT 10
""").pl()

### Most Popular Designer's Most Popular Patterns

#### By Favorites

In [ ]:
display(
    con.sql("""
        WITH most_popular_designer AS (
            SELECT
                id,
                name,
                favorites_count
            FROM clean_designers
            ORDER BY favorites_count DESC
            LIMIT 1
        )
        SELECT
            designer.name,
            pattern.name AS pattern_name,
            pattern.favorites_count AS num_favorites
        FROM most_popular_designer AS designer
        JOIN clean_patterns AS pattern
            ON designer.id = pattern.designer_id
        ORDER BY pattern.favorites_count DESC
        LIMIT 10
    """).pl()
)

#### By Projects

In [ ]:
display(
    con.sql("""
        WITH most_popular_designer AS (
            SELECT
                id,
                name,
                favorites_count
            FROM clean_designers
            ORDER BY favorites_count DESC
            LIMIT 1
        )
        SELECT
            designer.name,
            pattern.name AS pattern_name,
            pattern.projects_count AS num_projects
        FROM most_popular_designer AS designer
        JOIN clean_patterns AS pattern
            ON designer.id = pattern.designer_id
        ORDER BY pattern.projects_count DESC
        LIMIT 10
    """).pl()
)

#### Discussion

Domain knowledge: Projects seems a better representation.

### Most Popular Designer Patterns Per Year

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW popular_per_year AS
    WITH pattern_years AS (
        SELECT
            id,
            designer_id,
            YEAR(created_at) AS year
        FROM clean_patterns
    ),
    most_popular_designers AS (
        SELECT
            id,
            name
        FROM clean_designers
        ORDER BY favorites_count DESC
        LIMIT 10
    )
    SELECT
        designer.name AS designer_name,
        pattern.id AS pattern_id,
        pattern.year AS pattern_year,
    FROM most_popular_designers AS designer
    JOIN pattern_years AS pattern
        ON designer.id = pattern.designer_id
""")

display(
    con.sql("""
        PIVOT popular_per_year
        ON pattern_year
        USING COALESCE(COUNT(pattern_id), 0)
        GROUP BY designer_name
    """).pl()
)

In [ ]:

popular_per_year_df = con.sql("""
    SELECT
        designer_name,
        pattern_year,
        COUNT(pattern_id) AS pattern_count,
        SUM(COUNT(pattern_id)) OVER (
            PARTITION BY designer_name 
            ORDER BY pattern_year
        ) AS running_total
    FROM popular_per_year
    GROUP BY designer_name, pattern_year
    ORDER BY designer_name, pattern_year
""").df()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(
    data=popular_per_year_df,
    x="pattern_year",
    y="running_total",
    hue="designer_name",
    marker="o"
)
ax.xaxis.set_major_locator(MultipleLocator(2))

plt.title("Patterns Published per Year by Designer")
plt.xlabel("Year")
plt.ylabel("Pattern Count")

plt.legend(title="Designer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Most Prolific Designer Patterns Per Year

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW prolific_per_year AS
    WITH pattern_years AS (
        SELECT
            id,
            designer_id,
            YEAR(created_at) AS year
        FROM clean_patterns
    ),
    most_prolific_designers AS (
        SELECT
            id,
            name
        FROM clean_designers
        ORDER BY patterns_count DESC
        LIMIT 10
    )
    SELECT
        designer.name AS designer_name,
        pattern.id AS pattern_id,
        pattern.year AS pattern_year,
    FROM most_prolific_designers AS designer
    JOIN pattern_years AS pattern
        ON designer.id = pattern.designer_id
""")

display(
    con.sql("""
        PIVOT prolific_per_year
        ON pattern_year
        USING COALESCE(COUNT(pattern_id), 0)
        GROUP BY designer_name
    """).pl()
)

In [ ]:
prolific_per_year_df = con.sql("""
    SELECT
        designer_name,
        pattern_year,
        COUNT(pattern_id) AS pattern_count,
        SUM(COUNT(pattern_id)) OVER (
            PARTITION BY designer_name 
            ORDER BY pattern_year
        ) AS running_total
    FROM prolific_per_year
    GROUP BY designer_name, pattern_year
    ORDER BY designer_name, pattern_year
""").df()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(
    data=prolific_per_year_df,
    x="pattern_year",
    y="running_total",
    hue="designer_name",
    marker="o"
)
ax.xaxis.set_major_locator(MultipleLocator(2))

plt.title("Total Patterns Published by Designer")
plt.xlabel("Year")
plt.ylabel("Total Pattern Count")
# Move legend outside the plot so it doesn't cover the lines
plt.legend(title="Designer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
prolific_per_year_df = prolific_per_year_df.sort_values("pattern_year", ascending=True)
popular_per_year_df["pattern_year"] = popular_per_year_df["pattern_year"].astype(str)

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(
    data=popular_per_year_df,
    x="pattern_year",
    weights="pattern_count",
    hue="designer_name",
    multiple="stack",
    shrink=0.8,
    ax=ax
)
ax.xaxis.set_major_locator(MultipleLocator(1))

### Growth in patterns per year for most prolific ?

### Correlation Matrices

In [ ]:
corr_matrix_1 = con.sql("""
    SELECT
        patterns_count,
        favorites_count
    FROM clean_designers
""").df().corr()

corr_matrix_2 = con.sql("""
    SELECT
        knitting_pattern_count,
        crochet_pattern_count,
        favorites_count
    FROM clean_designers
""").df().corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
heatmap_settings = {
    "annot": True,
    "fmt": ".2f",
    "cmap": sns.blend_palette([
        "#e7f0f7",
        "#5a0e21"
    ], as_cmap=True),
    "square": True,
    "vmax": 1,
    "vmin": 0,
}

sns.heatmap(corr_matrix_1, ax=axes[0], **heatmap_settings)
axes[0].set_title("Raw Count (Pearson) Correlation")

sns.heatmap(corr_matrix_2, ax=axes[1], **heatmap_settings)
axes[1].set_title("Log-transformed (Spearman) Correlation")

plt.tight_layout()
plt.show()


## Challenges and Limitations

No major challenges were encountered in the wrangling and cleaning of the Designer dataset.

However, the Designer dataset may be of limited usefulness. The only unique value it provides is the `favorites_count` column. Other potentially useful columns are summary statistics that may quickly become dated.


### Outdated Summary Columns

In [ ]:
con.sql("""
    WITH differences AS (
        SELECT
            designer.name,
            designer.patterns_count,
            COUNT(pattern.*) AS num_patterns_dataset,
            designer.patterns_count - COUNT(pattern.*) AS diff
        FROM clean_designers AS designer
        JOIN clean_patterns AS pattern
            ON designer.id = pattern.designer_id
        GROUP BY designer.name, designer.patterns_count
        HAVING diff <> 0
    )
    SELECT
        COUNT(*) AS num_designers,
        SUM(ABS(diff)) AS total_pattern_diff,
        ROUND(AVG(ABS(diff)), 2) AS avg_pattern_diff
    FROM differences
""").pl()

Columns like `favorites_count` and `patterns_count` may quickly become dated. These numbers can change daily, but the scraper currently does not support periodically refreshing previously-fetched data.

Comparing the Designer `patterns_count` column against the Patterns dataset itself, we find over 3,600 Patterns are unaccounted for in the Designers dataset. This occurred over a span of 2 weeks, meaning approximately 186 new Patterns are uploaded each day. It is uncertain how often the `favorites_count` may change, but, given there are over 9 million users, we believe this value changes daily as well.

Updating the pipelines to periodically refresh data is an area for future improvement. We could also store counts like `favorites_count` in a time series, to analyze should the number ever decreases.